# LEROI example

This notebook shows the current LEROI workflow: read a Py-ART-compatible radar file, optionally mask a field, interpolate the field to a Cartesian grid, wrap the result in a Py-ART `Grid`, and plot one PPI next to one grid level.


In [ ]:
from pathlib import Path
import os

import leroi
import matplotlib.pyplot as plt
import numpy as np
import pyart


Set `LEROI_EXAMPLE_RADAR` to any local radar file that Py-ART can read before running the notebook. The example chooses a reflectivity-like field when one is available and otherwise falls back to the first field in the file.


In [ ]:
radar_path = Path(os.environ.get("LEROI_EXAMPLE_RADAR", "path/to/radar-file.nc"))
if not radar_path.exists():
    raise FileNotFoundError(
        "Set LEROI_EXAMPLE_RADAR to a Py-ART-readable radar file "
        "before running this notebook."
    )

radar = pyart.io.read(radar_path)
preferred_fields = ["reflectivity_horizontal", "reflectivity", "DBZH", "DBZ"]
field = next((name for name in preferred_fields if name in radar.fields), next(iter(radar.fields)))
fields = [field]
field


`mask_invalid_data` updates the radar object in place and returns it. Adjust `min_field` and `min_area` for the field and radar volume you are using.


In [ ]:
radar = leroi.mask_invalid_data(
    radar,
    field,
    add_to=fields,
    correlation_length=2000,
    min_field=0,
    min_area=50,
)


Define the output grid in `(z, y, x)` order. Coordinates are metres relative to the radar origin.


In [ ]:
grid_shape = (41, 301, 301)
grid_limits = ((0.0, 20000.0), (-150000.0, 150000.0), (-150000.0, 150000.0))
coords = [
    np.linspace(lower, upper, n)
    for (lower, upper), n in zip(grid_limits, grid_shape)
]


`leroi_interp` returns a Py-ART-style field dictionary. Leave `Rc=None` to estimate the radius of influence from the radar sweep spacing; increase `k` if LEROI warns that valid points are being left out.


In [ ]:
grid_fields = leroi.leroi_interp(
    radar,
    coords,
    field_names=fields,
    weight_type="Barnes",
    Rc=None,
    k=100,
    verbose=True,
)

grid = leroi.build_pyart_grid(radar, grid_fields, grid_shape, grid_limits)
grid


Plot a low-elevation PPI and the matching gridded field at one height level.


In [ ]:
z_index = 5
target_elevation = 0.5
sweep = int(np.nanargmin(np.abs(radar.fixed_angle["data"] - target_elevation)))

xr, yr, _ = radar.get_gate_x_y_z(sweep)
ppi_data = radar.get_field(sweep, field)
grid_data = grid.fields[field]["data"][z_index]

xlim = np.asarray(grid_limits[2]) / 1000.0
ylim = np.asarray(grid_limits[1]) / 1000.0
plot_kwargs = {"vmin": 0, "vmax": 50, "cmap": "turbo", "shading": "auto"}

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), constrained_layout=True)
mesh0 = axes[0].pcolormesh(xr / 1000.0, yr / 1000.0, ppi_data, **plot_kwargs)
mesh1 = axes[1].pcolormesh(coords[2] / 1000.0, coords[1] / 1000.0, grid_data, **plot_kwargs)

axes[0].set_title(f"PPI sweep {sweep}: {field}")
axes[1].set_title(f"Grid at z = {coords[0][z_index]:.0f} m")
for ax in axes:
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("x (km)")
    ax.set_ylabel("y (km)")
    ax.grid(True, alpha=0.3)

fig.colorbar(mesh1, ax=axes, label=grid.fields[field].get("units", field))
plt.show()
